# Offline component walkthrough

This notebook exercises only local, deterministic components. It does not call a model, Qdrant, Neo4j, or any external API. Run it from `component-lab/` with the repository root as the working directory, or set `PYTHONPATH` to `component-lab`.

In [ ]:
from pathlib import Path
import sys
import tempfile
from datetime import datetime, timezone

# VS Code may start the kernel with notebooks/ as its working directory.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from components.context import ContextBuilder, ContextConfig, ContextPacket
from components.memory import MemoryManager
from components.notes import NoteTool
from components.rag import RAGTool
from components.terminal import TerminalTool

In [ ]:
memory = MemoryManager()
memory.add('The user develops Pandas pipelines.', scope='user:demo', importance=0.9)
memory.add('Use categorical dtypes for repeated strings.', scope='user:demo', importance=0.8)
memory.consolidate(from_type='working', to_type='episodic', importance_threshold=0.7)
[(hit.record.id, hit.record.memory_type, round(hit.score, 3)) for hit in memory.search('Pandas dtypes', scope='user:demo')]

In [ ]:
rag = RAGTool()
rag.add_text((project_root / 'data/sample_knowledge.md').read_text(encoding='utf-8'), doc_id='sample')
skeleton = rag.ask_skeleton('How do I reduce Pandas memory?')
skeleton.prompt

In [ ]:
builder = ContextBuilder(
    memory=memory,
    rag=rag,
    memory_scope='user:demo',
    config=ContextConfig(max_tokens=300, reserve_ratio=0.2, min_relevance=0.0),
)
print(builder.build(
    'How can I reduce Pandas memory?',
    system_instructions='You are a data engineering advisor.',
    custom_packets=[ContextPacket(
        content='Task state: reviewing memory usage',
        timestamp=datetime.now(timezone.utc),
        relevance_score=0.9,
        metadata={'type': 'task_state'},
    )],
))

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    notes = NoteTool(Path(directory) / 'notes')
    note = notes.create('Memory audit', 'Review dtype choices.', note_type='action')
    print(notes.read(note.id))

terminal = TerminalTool(project_root)
print(terminal.run('ls').stdout.splitlines()[:5])